In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

26/02/26 12:49:53 WARN Utils: Your hostname, Rob resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/02/26 12:49:53 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/26 12:49:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/26 12:49:54 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/02/26 12:49:54 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [3]:
import pandas as pd

In [4]:
from pyspark.sql import types

In [25]:
green_schema = types.StructType([
    types.StructField("VendorID", types.IntegerType(), True),
    types.StructField("lpep_pickup_datetime", types.TimestampType(), True),
    types.StructField("lpep_dropoff_datetime", types.TimestampType(), True),
    types.StructField("store_and_fwd_flag", types.StringType(), True),
    types.StructField("RatecodeID", types.IntegerType(), True),
    types.StructField("PULocationID", types.LongType(), True),
    types.StructField("DOLocationID", types.LongType(), True),
    types.StructField("passenger_count", types.IntegerType(), True),
    types.StructField("trip_distance", types.DoubleType(), True),
    types.StructField("fare_amount", types.DoubleType(), True),
    types.StructField("extra", types.DoubleType(), True),
    types.StructField("mta_tax", types.DoubleType(), True),
    types.StructField("tip_amount", types.DoubleType(), True),
    types.StructField("tolls_amount", types.DoubleType(), True),
    types.StructField("ehail_fee", types.DoubleType(), True),
    types.StructField("improvement_surcharge", types.DoubleType(), True),
    types.StructField("total_amount", types.DoubleType(), True),
    types.StructField("payment_type", types.IntegerType(), True),
    types.StructField("trip_type", types.IntegerType(), True),
    types.StructField("congestion_surcharge", types.DoubleType(), True)
])

yellow_schema = types.StructType([
    types.StructField("VendorID", types.IntegerType(), True),
    types.StructField("tpep_pickup_datetime", types.TimestampType(), True),
    types.StructField("tpep_dropoff_datetime", types.TimestampType(), True),
    types.StructField("passenger_count", types.IntegerType(), True),
    types.StructField("trip_distance", types.DoubleType(), True),
    types.StructField("RatecodeID", types.IntegerType(), True),
    types.StructField("store_and_fwd_flag", types.StringType(), True),
    types.StructField("PULocationID", types.LongType(), True),
    types.StructField("DOLocationID", types.LongType(), True),
    types.StructField("payment_type", types.IntegerType(), True),
    types.StructField("fare_amount", types.DoubleType(), True),
    types.StructField("extra", types.DoubleType(), True),
    types.StructField("mta_tax", types.DoubleType(), True),
    types.StructField("tip_amount", types.DoubleType(), True),
    types.StructField("tolls_amount", types.DoubleType(), True),
    types.StructField("improvement_surcharge", types.DoubleType(), True),
    types.StructField("total_amount", types.DoubleType(), True),
    types.StructField("congestion_surcharge", types.DoubleType(), True)
])

In [6]:
year = 2020
base = Path("..")

for month in range(1, 13):
    print(f'processing data for {year}/{month}')

    from pathlib import Path
    
    base = Path("..")  # notebooks/ -> spark-project/
    
    input_path  = str(base / f"data/raw/green/{year}/{month:02d}/*.csv.gz")
    output_path = str(base / f"data/pq/green/{year}/{month:02d}/")

    df_green = spark.read \
        .option("header", "true") \
        .schema(green_schema) \
        .csv(input_path)

    (df_green
        .repartition(4)
        .write.mode("overwrite")
        .parquet(output_path)
    )

processing data for 2020/1


processing data for 2020/2


processing data for 2020/3
processing data for 2020/4
processing data for 2020/5
processing data for 2020/6
processing data for 2020/7
processing data for 2020/8
processing data for 2020/9
processing data for 2020/10
processing data for 2020/11
processing data for 2020/12


In [7]:
from pyspark.sql import functions as F
from pathlib import Path

year = 2021
base = Path("..")

for month in range(1, 13):
    print(f'processing data for {year}/{month:02d}')

    month_path = base / f"data/raw/green/{year}/{month:02d}"
    csv_path = str(month_path / "*.csv.gz")
    pq_path  = str(month_path / "*.parquet")

    if list(month_path.glob("*.csv.gz")):
        print("  -> reading CSV")
        df_green = (spark.read
            .option("header", "true")
            .schema(green_schema)
            .csv(csv_path)
        )

    elif list(month_path.glob("*.parquet")):
        print("  -> reading Parquet")
        df_green = spark.read.parquet(pq_path)

        # ✅ normalize types to match green_schema
        df_green = (df_green
            .withColumn("PULocationID", F.col("PULocationID").cast("int"))
            .withColumn("DOLocationID", F.col("DOLocationID").cast("int"))
        )

    else:
        print("  -> no files found, skipping")
        continue

    output_path = str(base / f"data/pq/green/{year}/{month:02d}/")

    (df_green
        .repartition(4)
        .write.mode("overwrite")
        .parquet(output_path)
    )

processing data for 2021/01
  -> reading CSV
processing data for 2021/02
  -> reading CSV
processing data for 2021/03
  -> reading CSV
processing data for 2021/04
  -> reading CSV
processing data for 2021/05
  -> reading CSV
processing data for 2021/06
  -> reading CSV
processing data for 2021/07
  -> reading CSV
processing data for 2021/08
  -> reading CSV
processing data for 2021/09
  -> reading Parquet
processing data for 2021/10
  -> reading Parquet
processing data for 2021/11
  -> reading Parquet
processing data for 2021/12
  -> reading Parquet


In [9]:
year = 2020

for month in range(1, 13):
    print(f'processing data for {year}/{month}')

    from pathlib import Path
    
    base = Path("..")  # notebooks/ -> spark-project/
    
    input_path  = str(base / f"data/raw/yellow/{year}/{month:02d}/*.csv.gz")
    output_path = str(base / f"data/pq/yellow/{year}/{month:02d}/")

    df_yellow = spark.read \
        .option("header", "true") \
        .schema(yellow_schema) \
        .csv(input_path)

    (df_yellow
        .repartition(4)
        .write.mode("overwrite")
        .parquet(output_path)
    )

processing data for 2020/1


processing data for 2020/2


processing data for 2020/3


processing data for 2020/4


processing data for 2020/5


...

processing data for 2020/8


processing data for 2020/9


processing data for 2020/10


processing data for 2020/11


processing data for 2020/12


In [12]:
from pathlib import Path
from pyspark.sql import functions as F

year = 2021
base = Path("..")

for month in range(1, 13):
    print(f"processing data for {year}/{month:02d}")

    month_path = base / f"data/raw/yellow/{year}/{month:02d}"
    csv_files = list(month_path.glob("*.csv.gz"))
    pq_files  = list(month_path.glob("*.parquet"))

    if pq_files:
        print("  -> reading Parquet")
        df_yellow = spark.read.parquet(str(month_path / "*.parquet"))

    elif csv_files:
        print("  -> reading CSV")
        df_yellow = (spark.read
            .option("header", "true")
            .schema(yellow_schema)
            .csv(str(month_path / "*.csv.gz"))
        )

    else:
        print("  -> no files found, skipping")
        continue

    # normalize to LONG everywhere
    df_yellow = (df_yellow
        .withColumn("PULocationID", F.col("PULocationID").cast("long"))
        .withColumn("DOLocationID", F.col("DOLocationID").cast("long"))
    )

    output_path = str(base / f"data/pq/yellow/{year}/{month:02d}/")
    (df_yellow
        .repartition(4)
        .write.mode("overwrite")
        .parquet(output_path)
    )

processing data for 2021/01
  -> reading CSV


processing data for 2021/02
  -> reading CSV


processing data for 2021/03
  -> reading CSV


processing data for 2021/04
  -> reading CSV


processing data for 2021/05
  -> reading CSV


...

processing data for 2021/08
  -> reading Parquet


processing data for 2021/09
  -> reading Parquet


processing data for 2021/10
  -> reading Parquet


processing data for 2021/11
  -> reading Parquet


processing data for 2021/12
  -> reading Parquet


In [13]:
# quick row count for Aug
spark.read.parquet("../data/pq/yellow/2021/08/").count()

2788757

In [14]:
spark.read.parquet("../data/pq/green/*/*") \
    .printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- lpep_pickup_datetime: timestamp (nullable = true)
 |-- lpep_dropoff_datetime: timestamp (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- ehail_fee: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- trip_type: integer (nullable = true)
 |-- congestion_surcharge: double (nullable = true)



In [15]:
spark.read.parquet("../data/pq/yellow/*/*") \
    .printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)



In [17]:
spark.read.parquet("../data/pq/green/*/*").count()

2495901

In [18]:
spark.read.parquet("../data/pq/yellow/*/*").count()

55552571

In [20]:
from pathlib import Path

base = Path("..")
for y in [2020, 2021]:
    for m in range(1, 13):
        p = base / f"data/pq/yellow/{y}/{m:02d}"
        parts = list(p.glob("part-*.parquet"))
        if len(parts) == 0:
            print("EMPTY FOLDER:", p)

In [21]:
from pathlib import Path

base = Path("..")
for y in [2020, 2021]:
    for m in range(1, 13):
        p = base / f"data/pq/green/{y}/{m:02d}"
        parts = list(p.glob("part-*.parquet"))
        if len(parts) == 0:
            print("EMPTY FOLDER:", p)

In [22]:
from pathlib import Path
from pyspark.sql import functions as F

base = Path("..")
year = 2021

for month in [9, 10, 11, 12]:
    month_path = base / f"data/raw/green/{year}/{month:02d}"
    out_path   = base / f"data/pq/green/{year}/{month:02d}"

    pq_files  = list(month_path.glob("*.parquet"))
    csv_files = list(month_path.glob("*.csv.gz"))

    if pq_files:
        print(f"{year}/{month:02d} -> reading raw Parquet")
        df = spark.read.parquet(str(month_path / "*.parquet"))
    elif csv_files:
        print(f"{year}/{month:02d} -> reading raw CSV")
        df = (spark.read.option("header", "true").schema(green_schema)
              .csv(str(month_path / "*.csv.gz")))
    else:
        print(f"{year}/{month:02d} -> no raw files, skipping")
        continue

    # normalize IDs to long
    df = (df
          .withColumn("PULocationID", F.col("PULocationID").cast("long"))
          .withColumn("DOLocationID", F.col("DOLocationID").cast("long")))

    (df
     .repartition(4)
     .write.mode("overwrite")
     .parquet(str(out_path)))

    print(f"wrote: {out_path}")

2021/09 -> reading raw Parquet
wrote: ../data/pq/green/2021/09
2021/10 -> reading raw Parquet
wrote: ../data/pq/green/2021/10
2021/11 -> reading raw Parquet
wrote: ../data/pq/green/2021/11
2021/12 -> reading raw Parquet
wrote: ../data/pq/green/2021/12


In [24]:
c09 = spark.read.parquet("../data/pq/green/2021/09").count()
c12 = spark.read.parquet("../data/pq/green/2021/12").count()
print("2021/09:", c09)
print("2021/12:", c12)

2021/09: 95709
2021/12: 99961
